# Symbolic Unconditioned Music Generation
**CSE 153/253 — Assignment 2**

## 1. Setup

In [11]:
!pip install pretty_midi

import pretty_midi
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import torch

## 2. Tokenizer

In [ ]:
# vocabulary constants
NOTE_ON_OFFSET    = 0
NOTE_OFF_OFFSET   = 128
TIME_SHIFT_OFFSET = 256
VELOCITY_OFFSET   = 356
INSTRUMENT_OFFSET = 388   # PIANO=388, VIOLIN=389

VOCAB_SIZE        = 390
N_TIME_SHIFT_BINS = 100
N_VELOCITY_BINS   = 32
TIME_STEP_MS      = 10
MAX_SHIFT_MS      = N_TIME_SHIFT_BINS * TIME_STEP_MS

PIANO_TOKEN  = 388
VIOLIN_TOKEN = 389

def velocity_to_bin(velocity):
    return min(velocity // 4, N_VELOCITY_BINS - 1)

def bin_to_velocity(bin_idx):
    return bin_idx * 4 + 2

In [ ]:
def get_instrument_token(instrument):
    name = instrument.name.lower()
    program = instrument.program
    # violin: program 40, viola: 41, cello: 42
    if program in range(40, 48) or "violin" in name:
        return VIOLIN_TOKEN
    return PIANO_TOKEN

def midi_to_tokens(midi_path):
    midi = pretty_midi.PrettyMIDI(str(midi_path))

    # Step 1: collect raw events, tagging each note with its instrument token
    raw_events = []
    for instrument in midi.instruments:
        inst_token = get_instrument_token(instrument)
        for note in instrument.notes:
            vbin = velocity_to_bin(note.velocity)
            raw_events.append((note.start, 'INSTRUMENT', inst_token))
            raw_events.append((note.start, 'VELOCITY',   vbin))
            raw_events.append((note.start, 'NOTE_ON',    note.pitch))
            raw_events.append((note.end,   'NOTE_OFF',   note.pitch))

    # Step 2: sort by time; for ties: INSTRUMENT → VELOCITY → NOTE_ON → NOTE_OFF
    type_order = {'INSTRUMENT': 0, 'VELOCITY': 1, 'NOTE_ON': 2, 'NOTE_OFF': 3}
    raw_events.sort(key=lambda e: (e[0], type_order[e[1]]))

    # Step 3: convert to tokens, inserting TIME_SHIFT tokens when the clock advances
    tokens = []
    current_time = 0.0

    for event_time, event_type, value in raw_events:
        delta_ms = int(round((event_time - current_time) * 1000))

        while delta_ms > 0:
            shift   = min(delta_ms, MAX_SHIFT_MS)
            n_steps = max(1, int(round(shift / TIME_STEP_MS)))
            n_steps = min(n_steps, N_TIME_SHIFT_BINS)
            tokens.append(TIME_SHIFT_OFFSET + n_steps - 1)
            delta_ms -= n_steps * TIME_STEP_MS

        current_time = event_time

        if event_type == 'NOTE_ON':
            tokens.append(NOTE_ON_OFFSET + value)
        elif event_type == 'NOTE_OFF':
            tokens.append(NOTE_OFF_OFFSET + value)
        elif event_type == 'VELOCITY':
            tokens.append(VELOCITY_OFFSET + value)
        elif event_type == 'INSTRUMENT':
            tokens.append(value)

    return tokens

In [ ]:
def tokens_to_midi(tokens, output_path):
    midi   = pretty_midi.PrettyMIDI()
    piano  = pretty_midi.Instrument(program=0,  name="Piano")
    violin = pretty_midi.Instrument(program=40, name="Violin")

    current_time        = 0.0
    current_velocity    = bin_to_velocity(16)
    current_instrument  = PIANO_TOKEN
    open_notes          = {}  # (pitch, instrument) → (start_time, velocity)

    for token in tokens:
        if token == PIANO_TOKEN or token == VIOLIN_TOKEN:
            current_instrument = token

        elif NOTE_ON_OFFSET <= token < NOTE_OFF_OFFSET:
            pitch = token - NOTE_ON_OFFSET
            open_notes[(pitch, current_instrument)] = (current_time, current_velocity)

        elif NOTE_OFF_OFFSET <= token < TIME_SHIFT_OFFSET:
            pitch = token - NOTE_OFF_OFFSET
            for inst in [current_instrument, PIANO_TOKEN, VIOLIN_TOKEN]:
                key = (pitch, inst)
                if key in open_notes:
                    start, vel = open_notes.pop(key)
                    end  = max(current_time, start + 0.01)
                    note = pretty_midi.Note(velocity=vel, pitch=pitch, start=start, end=end)
                    if inst == PIANO_TOKEN:
                        piano.notes.append(note)
                    else:
                        violin.notes.append(note)
                    break

        elif TIME_SHIFT_OFFSET <= token < VELOCITY_OFFSET:
            n_steps = token - TIME_SHIFT_OFFSET + 1
            current_time += n_steps * TIME_STEP_MS / 1000.0

        elif VELOCITY_OFFSET <= token < INSTRUMENT_OFFSET:
            current_velocity = bin_to_velocity(token - VELOCITY_OFFSET)

    # close any notes still open at the end
    for (pitch, inst), (start, vel) in open_notes.items():
        note = pretty_midi.Note(velocity=vel, pitch=pitch,
                                start=start, end=max(current_time, start + 0.01))
        if inst == PIANO_TOKEN:
            piano.notes.append(note)
        else:
            violin.notes.append(note)

    midi.instruments.append(piano)
    midi.instruments.append(violin)
    midi.write(str(output_path))
    print(f"Saved: {output_path} | piano notes: {len(piano.notes)} | violin notes: {len(violin.notes)}")

## 3. Sanity Check

In [ ]:
# Create a simple hand-crafted MIDI: piano plays C4, violin plays E4 simultaneously
test_midi  = pretty_midi.PrettyMIDI()
piano_inst = pretty_midi.Instrument(program=0,  name="Piano")
violin_inst= pretty_midi.Instrument(program=40, name="Violin")
piano_inst.notes.append( pretty_midi.Note(velocity=80, pitch=60, start=0.0, end=0.5))
violin_inst.notes.append(pretty_midi.Note(velocity=70, pitch=64, start=0.0, end=0.5))
piano_inst.notes.append( pretty_midi.Note(velocity=75, pitch=62, start=0.5, end=1.0))
violin_inst.notes.append(pretty_midi.Note(velocity=65, pitch=67, start=0.5, end=1.0))
test_midi.instruments.append(piano_inst)
test_midi.instruments.append(violin_inst)
test_midi.write("test_input.mid")

# Tokenize it
tokens = midi_to_tokens("test_input.mid")

# Print tokens with human-readable labels
def decode_token(t):
    if t == PIANO_TOKEN:
        return "INSTRUMENT(PIANO)"
    elif t == VIOLIN_TOKEN:
        return "INSTRUMENT(VIOLIN)"
    elif t < NOTE_OFF_OFFSET:
        return f"NOTE_ON({t})"
    elif t < TIME_SHIFT_OFFSET:
        return f"NOTE_OFF({t - NOTE_OFF_OFFSET})"
    elif t < VELOCITY_OFFSET:
        return f"TIME_SHIFT({(t - TIME_SHIFT_OFFSET + 1) * TIME_STEP_MS}ms)"
    else:
        return f"VELOCITY(bin={t - VELOCITY_OFFSET})"

print(f"Tokens ({len(tokens)} total):")
for t in tokens:
    print(f"  {t:3d}  →  {decode_token(t)}")

# Round-trip check
tokens_to_midi(tokens, "test_roundtrip.mid")
tokens_rt = midi_to_tokens("test_roundtrip.mid")
print(f"\nRound-trip match: {tokens == tokens_rt}")

## 4. Data Loading

In [ ]:
import random

DATASET_DIR = Path("lmd_full")

def has_piano_and_violin(midi_path):
    try:
        midi = pretty_midi.PrettyMIDI(str(midi_path))
        programs = [i.program for i in midi.instruments if not i.is_drum]
        has_piano  = any(p in range(0, 8) for p in programs)   # piano family
        has_violin = any(p in range(40, 48) for p in programs) # string family
        return has_piano and has_violin
    except:
        return False

# scan all MIDI files
all_midi = list(DATASET_DIR.rglob("*.mid"))
print(f"Total MIDI files: {len(all_midi):,}")

print("Filtering for piano + violin files...")
filtered = [p for i, p in enumerate(all_midi) if has_piano_and_violin(p)]
print(f"Piano + violin files: {len(filtered):,}")

# shuffle and split 90/10 into train/val
random.seed(42)
random.shuffle(filtered)
split      = int(0.9 * len(filtered))
train_files = filtered[:split]
val_files   = filtered[split:]

print(f"Train files : {len(train_files):,}")
print(f"Val files   : {len(val_files):,}")

In [17]:
def tokenize_dataset(file_list, output_path):
    all_tokens = []
    for i, path in enumerate(file_list):
        try:
            tokens = midi_to_tokens(path)
            all_tokens.extend(tokens)
        except Exception as e:
            print(f"Skipping {path.name}: {e}")
        if (i + 1) % 100 == 0:
            print(f"  {i+1}/{len(file_list)} files processed")

    arr = np.array(all_tokens, dtype=np.int16)
    np.save(output_path, arr)
    print(f"Saved {len(arr):,} tokens to {output_path}")
    return arr

print("Tokenizing train set...")
train_tokens = tokenize_dataset(train_files, "train_tokens.npy")

print("\nTokenizing validation set...")
val_tokens = tokenize_dataset(val_files, "val_tokens.npy")

Tokenizing train set...
  100/962 files processed
  200/962 files processed
  300/962 files processed
  400/962 files processed
  500/962 files processed
  600/962 files processed
  700/962 files processed
  800/962 files processed
  900/962 files processed
Saved 31,423,621 tokens to train_tokens.npy

Tokenizing validation set...
  100/137 files processed
Saved 3,554,013 tokens to val_tokens.npy


In [18]:
from torch.utils.data import Dataset, DataLoader

SEQ_LEN = 512

class MusicDataset(Dataset):
    def __init__(self, tokens):
        self.tokens = torch.tensor(tokens.astype(np.int64))

    def __len__(self):
        # each sample needs SEQ_LEN input tokens + 1 target token
        return len(self.tokens) - SEQ_LEN

    def __getitem__(self, idx):
        x = self.tokens[idx : idx + SEQ_LEN]          # input
        y = self.tokens[idx + 1 : idx + SEQ_LEN + 1]  # target (shifted by 1)
        return x, y

train_dataset = MusicDataset(train_tokens)
val_dataset   = MusicDataset(val_tokens)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False)

print(f"Train samples : {len(train_dataset):,}")
print(f"Val samples   : {len(val_dataset):,}")
print(f"Train batches : {len(train_loader):,}")

Train samples : 31,423,109
Val samples   : 3,553,501
Train batches : 981,973


## 5. Model

In [ ]:
import torch.nn as nn

class MusicTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, num_layers, ff_dim, dropout, max_seq_len):
        super().__init__()
        self.embedding    = nn.Embedding(vocab_size, embed_dim)
        self.pos_encoding = nn.Embedding(max_seq_len, embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads,
            dim_feedforward=ff_dim, dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.fc          = nn.Linear(embed_dim, vocab_size)
        self.max_seq_len = max_seq_len

    def forward(self, x):
        seq_len   = x.size(1)
        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        mask      = nn.Transformer.generate_square_subsequent_mask(seq_len, device=x.device)
        x = self.embedding(x) + self.pos_encoding(positions)
        x = self.transformer(x, mask=mask, is_causal=True)
        return self.fc(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = MusicTransformer(
    vocab_size  = VOCAB_SIZE,
    embed_dim   = 256,
    num_heads   = 8,
    num_layers  = 4,
    ff_dim      = 1024,
    dropout     = 0.1,
    max_seq_len = 512,
).to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

In [ ]:
import torch.nn.functional as F

LEARNING_RATE = 0.001
MAX_STEPS     = 50
VAL_EVERY     = 1000
SAVE_EVERY    = 5000

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

def evaluate(model, loader, max_batches=50):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            if i >= max_batches:
                break
            x, y   = x.to(device), y.to(device)
            logits = model(x)
            loss   = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1))
            total_loss += loss.item()
    model.train()
    return total_loss / min(max_batches, len(loader))

train_iter   = iter(train_loader)
step         = 0
train_losses = []
val_losses   = []

model.train()
while step < MAX_STEPS:
    try:
        x, y = next(train_iter)
    except StopIteration:
        train_iter = iter(train_loader)
        x, y = next(train_iter)

    x, y   = x.to(device), y.to(device)

    optimizer.zero_grad()
    logits = model(x)
    loss   = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    train_losses.append(loss.item())
    step += 1

    if step % VAL_EVERY == 0:
        val_loss = evaluate(model, val_loader)
        val_losses.append((step, val_loss))
        print(f"Step {step:6d} | train loss: {loss.item():.4f} | val loss: {val_loss:.4f}")

    if step % SAVE_EVERY == 0:
        torch.save(model.state_dict(), f"checkpoint_step{step}.pt")
        print(f"  → checkpoint saved")

print("Training complete.")

## 7. Generation

In [ ]:
def generate(model, seed_tokens, n_tokens=1000, temperature=1.0):
    model.eval()
    tokens = list(seed_tokens)

    with torch.no_grad():
        for _ in range(n_tokens):
            # use the last max_seq_len tokens as context
            context = tokens[-model.max_seq_len:]
            x       = torch.tensor(context, dtype=torch.long).unsqueeze(0).to(device)
            logits  = model(x)
            logits  = logits[:, -1, :] / temperature
            probs   = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, 1).item()
            tokens.append(next_token)

    return tokens[len(seed_tokens):]

# use the first 100 tokens of training data as seed
seed      = train_tokens[:100].astype(np.int64).tolist()
generated = generate(model, seed, n_tokens=2000, temperature=1.0)
tokens_to_midi(generated, "symbolic_unconditioned.mid")
print(f"Generated {len(generated)} tokens")

## 8. Evaluation